# DirectMultiStep, Taken Apart: a Hands-On Primer
**Series B: Deep Learning for Molecules, Post 6d** &nbsp;|&nbsp; companion *primer* notebook

DirectMultiStep (DMS; Shee, Morgunov, Li and Batista, *J. Chem. Inf. Model.* **65**, 3903, 2025; code at github.com/batistagroup/DirectMultiStep) generates a **complete multi-step retrosynthesis route as one string**: a transformer reads the target molecule (optionally also a starting material and a number of steps) and writes the whole synthesis tree, token by token.

> **What this notebook is, and is not.** It rebuilds the *machinery around the model* in NumPy: route trees and their string form, tokenization, permutation augmentation, the length-normalized score used in beam search, a sparsely-gated mixture-of-experts layer, the architecture's parameter counts, the learning-rate schedule, and an analysis of the paper's tables. It does **not** train or run a DirectMultiStep transformer (that needs PyTorch, the PaRoutes training data and a GPU) and does not reproduce its accuracy. For that, use the authors' package (`pip install directmultistep`, checkpoints via `download_files.sh`). The routes below are hand-written illustrations; SMILES are checked for *syntax* only (no RDKit here), not for chemistry.

| Section | What you do |
|---|---|
| 1 | represent a route as a tree; compute its number of steps, leaves and convergence |
| 2 | serialize a route to a string and tokenize it; measure its length |
| 3 | permutation augmentation, and the data arithmetic of the paper |
| 4 | why raw log-likelihood favours short routes, and what length normalization does |
| 5 | a sparsely-gated mixture-of-experts layer; Mono and Duo modes; parameter counts of all variants |
| 6 | the learning-rate schedule and input masking of the training recipe |
| 7 | analyse the paper's accuracy tables (speed-ups, rank-1 share, value of extra inputs) |
| 8 | route length: how many test routes are long, and how sure can we be? |
| 9 | the search-space argument |

## 0. Setup

In [ ]:
import ast, math, re, itertools
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. A route is a tree
The paper's convention: the **target** is the root; a **starting material** is a leaf (no further precursors are sought); the **number of steps** is the height of the tree (the largest number of reactions between a starting material and the target). A node is a dictionary with the molecule's SMILES and, unless it is a leaf, a list of **children**: the reactants of the reaction that makes it.
Here is an *illustrative* two-step route of the kind shown for the drug vonoprazan in the paper's Figure 5 (hand-written for this notebook, not copied from the paper): a pyrrole aldehyde is sulfonylated with pyridine-3-sulfonyl chloride, then converted to the target by reductive amination with methylamine.

In [ ]:
route = {"smiles": "CNCc1cc(-c2ccccc2F)n(S(=O)(=O)c2cccnc2)c1",                      # target (the SMILES of the README's example)
         "children": [{"smiles": "O=Cc1cc(-c2ccccc2F)n(S(=O)(=O)c2cccnc2)c1",           # intermediate aldehyde
                       "children": [{"smiles": "O=Cc1c[nH]c(-c2ccccc2F)c1"},               # starting material
                                    {"smiles": "O=S(=O)(Cl)c1cccnc1"}]},                    # starting material
                      {"smiles": "CN"}]}                                                   # starting material (methylamine)

is_leaf = lambda node: not node.get("children")
def n_steps(node):       return 0 if is_leaf(node) else 1 + max(n_steps(c) for c in node["children"])      # tree height
def n_reactions(node):   return 0 if is_leaf(node) else 1 + sum(n_reactions(c) for c in node["children"])
def leaves(node):        return [node["smiles"]] if is_leaf(node) else [s for c in node["children"] for s in leaves(c)]
def convergent(node):                                                        # the paper: some node has at least two children that are not leaves
    if is_leaf(node): return False
    return sum(not is_leaf(c) for c in node["children"]) >= 2 or any(convergent(c) for c in node["children"])

print("number of steps (tree height):", n_steps(route), "| reactions:", n_reactions(route), "| starting materials:", leaves(route))
print("convergent route:", convergent(route))

# a tiny SYNTAX check for SMILES (no chemistry): known tokens only, balanced parentheses and brackets, ring-closure digits in pairs
SMILES_TOKEN = re.compile(r"\[[^\]]+\]|Br|Cl|%\d\d|[BCNOPSFI]|[bcnops]|[()=#\-+\\/:~@?>*$.]|\d")
def smiles_syntax_ok(s):
    toks = SMILES_TOKEN.findall(s)
    if "".join(toks) != s: return False
    depth = 0
    for t in toks:
        depth += (t == "(") - (t == ")")
        if depth < 0: return False
    rings = Counter(t for t in toks if t.isdigit() or t.startswith("%"))
    return depth == 0 and all(v % 2 == 0 for v in rings.values())

def all_nodes(node): return [node["smiles"]] + [s for c in node.get("children", []) for s in all_nodes(c)]
print("all SMILES pass the syntax check:", all(smiles_syntax_ok(s) for s in all_nodes(route)))
print("a broken SMILES fails it:", smiles_syntax_ok("CC(C1CC"), smiles_syntax_ok("c1ccccc"))

## 2. From tree to string to tokens
The model writes the route as text: "removing space and newline characters creates a string representation", tokenized **one character at a time**, except that the tree's delimiters and keys are single tokens (the paper lists `'smiles'`, `'children'`, `,`, `[`, `]`, `{`, `}`), plus start, end and padding tokens; the full vocabulary of the paper has **52** tokens. The exact tokenization is in the authors' code (which we did not read), so below is *our reading* of the description.
**Stop and predict:** roughly how many tokens does a 2-step route with three starting materials need?

In [ ]:
def serialize(node):
    return repr(node).replace(" ", "")                                       # a Python-dict-like string with spaces and newlines removed

TOKEN = re.compile(r"'smiles'|'children'|.", re.S)
def tokenize(s):                                                              # the two keys are single tokens; every other character is one token (as in the paper)
    return [m.group(0) for m in TOKEN.finditer(s)]

s = serialize(route)
print(s)
toks = tokenize(s)
print(f"\nstring length {len(s)} characters -> {len(toks)} tokens")
print("round trip (parse the string back into a tree):", ast.literal_eval(s) == route)
print("first 25 tokens:", toks[:25])
vocab_toy = sorted(set(toks)); print(f"distinct tokens in this one route: {len(vocab_toy)} (the paper's full vocabulary: 52, including start, end and padding)")
n_smiles_chars = sum(len(x) for x in all_nodes(route))
print(f"{n_smiles_chars} of the {len(toks)} tokens are SMILES characters; the other {len(toks) - n_smiles_chars} are structure (keys, quotes, colons, commas, brackets, braces): about {(len(toks) - n_smiles_chars) / len(all_nodes(route)):.0f} per node")
tok_per_reaction = len(toks) / n_reactions(route)
print(f"about {tok_per_reaction:.0f} tokens per reaction in this example")
print("paper: the largest route in its data has 1074 tokens; the largest target 145; the largest starting material 135")

## 3. Permutations, and the data arithmetic
The two children of a reaction can be written in either order, so one route has several strings. The paper **augments** its training set with 2 random permutations per route and removes permutations of the test routes from the training set. For a route in which node $v$ has $k_v$ distinct children there are $\prod_v k_v!$ strings.

In [ ]:
def permutations_of(node):
    if is_leaf(node): return [node]
    child_variants = [permutations_of(c) for c in node["children"]]
    out = []
    for order in itertools.permutations(range(len(node["children"]))):
        for combo in itertools.product(*[child_variants[i] for i in order]):
            out.append({"smiles": node["smiles"], "children": list(combo)})
    return out

def canonical(node):                                                         # order-independent form: sort the children by their own canonical form
    if is_leaf(node): return {"smiles": node["smiles"]}
    kids = sorted((canonical(c) for c in node["children"]), key=lambda d: repr(d))
    return {"smiles": node["smiles"], "children": kids}

variants = permutations_of(route)
strings = {serialize(v) for v in variants}
print(f"{len(variants)} permuted trees = prod of k_v! = 2! x 2! ; distinct strings: {len(strings)}")
print("all variants are the same route once children are sorted:", len({repr(canonical(v)) for v in variants}) == 1)

# arithmetic of the paper's data curation
routes_train, inputs_noSM, inputs_SM = 163_689, 432_684, 1_340_243
print(f"\ntraining routes {routes_train:,}; x3 (original + 2 permutations) = {3 * routes_train:,}")
print(f"inputs actually used without starting material: {inputs_noSM:,} -> {inputs_noSM / routes_train:.3f} strings per route; "
      f"{100 * (1 - inputs_noSM / (3 * routes_train)):.1f} % fewer than 3 per route")
print("(permuting children only changes the string of a route that has a node with two or more different children, so identical strings are removed as duplicates; this is our reading of the numbers)")
print(f"inputs with starting material: {inputs_SM:,} -> {inputs_SM / inputs_noSM:.2f} starting materials per route string, on average (one training entry for each leaf)")
print(f"hidden dimension 256 vs vocabulary 52: {256 / 52:.1f}x   (the paper: 'almost 5x')")

## 4. Why a length-normalized score?
A language model scores a candidate route by the sum of the log-probabilities of its tokens. Every token multiplies the probability by a number below 1, so **longer routes always score lower**, whatever their content, and the beam would fill with short, simple routes. DMS therefore normalizes the cumulative log-likelihood by the length at the time each new token is sampled (so a candidate of $t$ tokens is ranked by $\frac1t\sum_{i\le t}\ln p_i$).
We see the effect exactly on a tiny language: 3 symbols plus an end token, where the end token has probability 0.25 at every step. We enumerate every sequence of up to 8 symbols.

In [ ]:
rng4 = np.random.default_rng(1)
nt, END, p_end = 3, 3, 0.25
P = np.zeros((nt + 1, nt + 1))                                   # rows: previous symbol (row 3 = start); columns: symbols 0-2 and END
for r in range(nt + 1):
    P[r, :nt] = rng4.dirichlet(np.ones(nt) * 1.2) * (1 - p_end); P[r, END] = p_end

def log_prob(seq):                                               # includes the final END step
    prev, lp = nt, 0.0
    for t in seq: lp += np.log(P[prev, t]); prev = t
    return lp + np.log(P[prev, END])

all_seq = [s for L in range(0, 9) for s in itertools.product(range(nt), repeat=L)]
raw = np.array([log_prob(s) for s in all_seq]); length = np.array([len(s) + 1 for s in all_seq])      # +1: the END token
norm = raw / length
print(f"{len(all_seq):,} sequences enumerated (total probability {np.exp(raw).sum():.3f}; the rest has length > 8)\n")
for name, score in (("raw log-likelihood", raw), ("length-normalized", norm)):
    top = np.argsort(-score)[:5]
    print(f"{name:20s} top-5 sequence lengths: {[len(all_seq[i]) for i in top]}   (mean {np.mean([len(all_seq[i]) for i in top]):.1f})")
print("\nraw scores put the SHORTEST sequences first; normalization removes that bias, and then favours sequences made of locally probable steps, here the longest allowed.")
print("A trained model emits its end token when the route is complete, which is what keeps normalized beams from running on forever.")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.scatter(length - 1, raw, s=6, color=COLORS["red"], label="raw log-likelihood"); ax.scatter(length - 1, norm * 8, s=6, color=COLORS["blue"], label="normalized (x 8, for scale)")
ax.set(xlabel="sequence length", ylabel="score"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. Mixture-of-experts layers
In DMS-Flex and DMS-Wide the two-layer feed-forward network that follows each attention block is replaced by an **ensemble of feed-forward "experts"** and a **router**: for each token, the router scores the experts, keeps the $k$ best, and adds their outputs with weights given by the router. (Here "experts" are sub-networks of the transformer: unrelated to the "experts" of MOSAIC.) A model trained with $k=2$ can be run with $k=1$ (**Mono**) or $k=2$ (**Duo**).

In [ ]:
def moe_ffn(x, W_gate, experts, k):
    """x: (tokens, d); experts: list of (W1, b1, W2, b2); router keeps the k highest scores and softmax-normalizes them."""
    logits = x @ W_gate                                            # (tokens, n_experts)
    idx = np.argsort(-logits, 1)[:, :k]; sel = np.take_along_axis(logits, idx, 1)
    w = np.exp(sel - sel.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    out = np.zeros_like(x)
    for j, (W1, b1, W2, b2) in enumerate(experts):
        h = np.maximum(x @ W1 + b1, 0) @ W2 + b2                  # this expert's output for every token (computed here for clarity)
        out += h * (w * (idx == j)).sum(1, keepdims=True)         # only tokens routed to expert j receive its output
    return out, idx

d, f, n_exp = 8, 24, 3
r5 = np.random.default_rng(0)
experts = [(r5.normal(size=(d, f)) * .3, r5.normal(size=f) * .1, r5.normal(size=(f, d)) * .3, r5.normal(size=d) * .1) for _ in range(n_exp)]
Wg = r5.normal(size=(d, n_exp)); x = r5.normal(size=(5, d))

y1, i1 = moe_ffn(x, Wg, experts, 1); y2, i2 = moe_ffn(x, Wg, experts, 2); y3, _ = moe_ffn(x, Wg, experts, 3)
dense = [np.maximum(x @ W1 + b1, 0) @ W2 + b2 for W1, b1, W2, b2 in experts]
print("k = 1 reduces to the top expert alone:", np.allclose(y1, np.stack([dense[j][t] for t, j in enumerate(i1[:, 0])])))
print("single expert, k = 1, is an ordinary feed-forward layer:", np.allclose(moe_ffn(x, Wg[:, :1], experts[:1], 1)[0], dense[0]))
print("each token's routing weights sum to 1; tokens routed to experts", i2.tolist())

In [ ]:
# parameter counts of all DMS variants from the paper's architecture table (hidden 256, feed-forward multiplier 3, vocabulary 52)
def dms_params(enc, dec, n_experts, active, d=256, ff=3, vocab=52):
    f = ff * d
    attn = 4 * d * d + 4 * d
    ffn = d * f + f + f * d + d
    ln = 2 * d
    router = d * n_experts + n_experts if n_experts > 1 else 0
    enc_layer = lambda k: attn + k * ffn + router + 2 * ln
    dec_layer = lambda k: 2 * attn + k * ffn + router + 3 * ln                 # self-attention + cross-attention
    fixed = vocab * d + d * vocab + vocab + 2 * ln                              # embedding, output layer, final norms
    total = enc * enc_layer(n_experts) + dec * dec_layer(n_experts) + fixed
    act = enc * enc_layer(active) + dec * dec_layer(active) + fixed
    return total, act

dms_variants = {"Flash": (6, 6, 1, 1, "10M"), "Explorer": (6, 6, 3, 2, "19M"), "Explorer XL": (8, 24, 3, 2, "50M"),
            "Flex": (6, 6, 3, 2, "19M"), "Deep": (12, 36, 1, 1, "41M in Table 3 (38M in the text)"), "Wide": (12, 12, 3, 2, "38M")}
rows = []
for name, (e, dcl, ne, ac, stated) in dms_variants.items():
    t, a = dms_params(e, dcl, ne, ac); rows.append((name, e, dcl, ne, ac, f"{t / 1e6:.2f} M", f"{a / 1e6:.2f} M", f"{100 * a / t:.0f} %", stated))
print(pd.DataFrame(rows, columns=["variant", "enc layers", "dec layers", "experts", "active", "total (our count)", "active per token", "active share", "paper"]).to_string(index=False))
print("\nour counts agree with the paper's sizes within 5 %; for DMS-Deep the count (41.1 M) matches Table 3's 41M, so the '38M' for Deep in the text of Section 3.3.2 looks like a typo.")

In [ ]:
# Mono vs Duo on a small trained mixture of experts: 3 experts, a 1-D regression with three regimes, router trained with k = 2
rg = np.random.default_rng(1)
target = lambda x: np.where(x < 1, np.sin(9 * x), np.where(x < 2, 1.0 - 6 * (x - 1.5) ** 2, np.cos(10 * (x - 2)) * (x - 1.8)))
Xa = np.sort(rg.uniform(0, 3, 900)); ya = target(Xa) + rg.normal(0, 0.05, 900); tr = np.arange(900) % 2 == 0
Xt, yt, Xv, yv = Xa[tr], ya[tr], Xa[~tr], ya[~tr]
E_, H = 3, 8; n_par = E_ * (3 * H + 1) + 2 * E_

def unpack(th):
    ex = [(th[i:i + H], th[i + H:i + 2 * H], th[i + 2 * H:i + 3 * H], th[i + 3 * H]) for i in range(0, E_ * (3 * H + 1), 3 * H + 1)]
    return ex, th[E_ * (3 * H + 1):E_ * (3 * H + 1) + E_], th[-E_:]

def moe1d(th, x, k):
    ex, gW, gb = unpack(th); logit = np.outer(x, gW) + gb
    out = np.stack([np.tanh(np.outer(x, W1) + b1) @ W2 + b2 for W1, b1, W2, b2 in ex], 1)
    idx = np.argsort(-logit, 1)[:, :k]; sel = np.take_along_axis(logit, idx, 1); w = np.exp(sel - sel.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return (w * np.take_along_axis(out, idx, 1)).sum(1), idx

def init_moe(seed):
    th = np.random.default_rng(seed).normal(size=n_par)
    th[-2 * E_:-E_] = [4, 12, 20]; th[-E_:] = [-1, -9, -25]                  # router starts with a territory for each expert (centres 0.5, 1.5, 2.5)
    return th

rmse = lambda th, k: float(np.sqrt(np.mean((moe1d(th, Xv, k)[0] - yv) ** 2)))
fit_k = lambda k, seed: minimize(lambda t: np.mean((moe1d(t, Xt, k)[0] - yt) ** 2), init_moe(seed), method="L-BFGS-B", options=dict(maxiter=500)).x
fits2 = [fit_k(2, s) for s in range(6)]
table = np.array([[rmse(th, k) for k in (1, 2, 3)] for th in fits2])
print("trained with k = 2, six random starts; validation RMSE when run with k = 1 / 2 / 3 active experts:")
print(pd.DataFrame(table, columns=["k=1 (Mono)", "k=2 (Duo)", "k=3"]).round(3).to_string())
print(f"\nmedian: {np.median(table, 0).round(3)} | k=1 / k=2 error ratio: median {np.median(table[:, 0] / table[:, 1]):.2f}, range {np.min(table[:, 0] / table[:, 1]):.2f} to {np.max(table[:, 0] / table[:, 1]):.2f}")
fits1 = [fit_k(1, s) for s in range(6)]
print(f"trained AND run with k = 1: median RMSE {np.median([rmse(th, 1) for th in fits1]):.3f}")
print("a model trained for two active experts cannot simply be run with one (the experts have learned to be used in pairs); running it with three adds nothing.")
best = fits2[int(np.argmin(table[:, 1]))]; _, idx = moe1d(best, Xv, 2)
for lo, hi in ((0, 1), (1, 2), (2, 3)):
    m = (Xv >= lo) & (Xv < hi); print(f"   top-1 expert share in x in [{lo},{hi}):", [round(float(np.mean(idx[m, 0] == e)), 2) for e in range(E_)])
print("The paper's Mono mode (trained with 2, run with 1) scores 0.32 Top-1 on n1 against 0.43 for Duo. We make NO accuracy claim for MoE here; the paper itself reports a gain of only 1-2 points over a same-depth dense model.")

## 6. The training recipe
Typed from the paper (Section 5.3): AdamW; the learning rate **warms up from 0 to a peak** over the first 10 % of the steps and then follows a **cosine decay to 10 % of the peak** over 20 epochs; peaks $3.5\times10^{-4}$ (Flex, Wide), $3\times10^{-4}$ (Flash), $10^{-4}$ (Deep); dropout 10 %; gradient clipping at 1.0; and **each token of the encoder input has a 5 % chance of being masked**.

In [ ]:
def lr_schedule(t, total, peak, warm=0.1, floor=0.1):
    t0 = warm * total
    if t < t0: return peak * t / t0
    return peak * (floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * (t - t0) / (total - t0))))

T = 1000; steps = np.arange(T + 1)
plt.figure(figsize=(7, 3.4))
for name, peak in (("Flex / Wide", 3.5e-4), ("Flash", 3e-4), ("Deep", 1e-4)):
    plt.plot(steps / T, [lr_schedule(t, T, peak) for t in steps], label=f"{name}, peak {peak:.1e}")
plt.xlabel("fraction of training"); plt.ylabel("learning rate"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("learning rate at 0 %, 10 %, 55 %, 100 % of training for peak 3.5e-4:", [f"{lr_schedule(f * T, T, 3.5e-4):.2e}" for f in (0, 0.1, 0.55, 1.0)])

mask_rng = np.random.default_rng(0); tokens = mask_rng.integers(0, 52, size=200_000); masked = mask_rng.random(len(tokens)) < 0.05
print(f"fraction of input tokens masked over {len(tokens):,} tokens: {masked.mean():.4f} (target 0.05)")
print("masking forces the encoder to cope with missing characters, a regularizer; the model also tolerates missing starting-material information in use")

## 7. Reading the paper's accuracy tables
Top-$K$ accuracy: the experimentally known route is among the first $K$ ranked candidates. Typed from Tables 3 and 4 (10,000 test routes each): the search-based baselines (single-step model plus tree search, from PaRoutes) and the DMS variants; "SM (gen.)" means the starting material is given at generation, "Steps" that the number of steps is given.

In [ ]:
cols = ["Top-1", "Top-5", "Top-10"]
base_n1 = pd.DataFrame({"MCTS": [0.17, 0.46, 0.49], "Retro*": [0.15, None, 0.45], "DFPN": [0.11, 0.17, 0.17]}, index=cols).T
base_n5 = pd.DataFrame({"MCTS": [0.10, 0.28, 0.33], "Retro*": [0.10, 0.30, 0.36], "DFPN": [0.05, 0.07, 0.07]}, index=cols).T
dms_cols = ["params", "steps", "SM_train", "SM_gen", "Top-1", "Top-2", "Top-3", "Top-4", "Top-5", "Top-10"]
dms_n1 = pd.DataFrame([("Explorer-XL", "50M", 0, 0, 0, .32, .40, .44, .45, .46, .48), ("Deep", "41M", 1, 0, 0, .36, .44, .47, .49, .50, .52),
                       ("Explorer", "19M", 0, 1, 1, .36, .45, .48, .51, .52, .55), ("Explorer, no SM", "19M", 0, 1, 0, .27, .33, .36, .37, .39, .40),
                       ("Flash", "10M", 1, 1, 1, .41, .50, .54, .56, .58, .60), ("Flash, no SM", "10M", 1, 1, 0, .32, .39, .42, .44, .45, .47),
                       ("Wide", "38M", 1, 0, 0, .38, .46, .49, .50, .51, .53), ("Flex (Mono)", "19M", 1, 1, 1, .32, .40, .43, .44, .45, .48),
                       ("Flex (Duo)", "19M", 1, 1, 1, .43, .52, .55, .57, .58, .61), ("Flex (Duo), no SM", "19M", 1, 1, 0, .34, .41, .44, .45, .46, .47)],
                      columns=["model"] + dms_cols).set_index("model")
dms_n5 = pd.DataFrame([("Explorer-XL", .31, .38, .41, .42, .43, .45), ("Deep", .33, .40, .43, .45, .46, .48), ("Explorer", .33, .40, .43, .45, .46, .49),
                       ("Explorer, no SM", .25, .31, .34, .35, .36, .37), ("Flash", .37, .44, .47, .49, .51, .53), ("Flash, no SM", .30, .36, .39, .40, .41, .43),
                       ("Wide", .35, .42, .44, .46, .47, .49), ("Flex (Mono)", .28, .34, .36, .38, .39, .41), ("Flex (Duo)", .39, .46, .49, .51, .52, .54),
                       ("Flex (Duo), no SM", .32, .38, .40, .42, .42, .44)], columns=["model", "Top-1", "Top-2", "Top-3", "Top-4", "Top-5", "Top-10"]).set_index("model")
topk = ["Top-1", "Top-2", "Top-3", "Top-4", "Top-5", "Top-10"]
print("Top-K is cumulative, hence non-decreasing in K in every row:", bool((np.diff(dms_n1[topk].values, axis=1) >= -1e-9).all() and (np.diff(dms_n5[topk].values, axis=1) >= -1e-9).all()))

best_base_1, best_base_5 = base_n1["Top-1"].max(), base_n5["Top-1"].max()
print(f"\nTop-1 improvement over the best search baseline: Explorer-XL x{dms_n1.loc['Explorer-XL', 'Top-1'] / best_base_1:.2f} (n1), x{dms_n5.loc['Explorer-XL', 'Top-1'] / best_base_5:.2f} (n5);"
      f" Flex (Duo) x{dms_n1.loc['Flex (Duo)', 'Top-1'] / best_base_1:.2f} (n1), x{dms_n5.loc['Flex (Duo)', 'Top-1'] / best_base_5:.2f} (n5)")
print("(abstract: 1.9x and 3.1x for Explorer-XL; Section 4: 2.5x and 3.9x for Flex (Duo); the text of Section 3.3 says 1.8x for n1, our ratio rounds to 1.9)")
rel = pd.DataFrame({"Top-1 n1": dms_n1.loc["Flex (Duo)", "Top-1"] / dms_n1.loc["Explorer-XL", "Top-1"] - 1, "Top-1 n5": dms_n5.loc["Flex (Duo)", "Top-1"] / dms_n5.loc["Explorer-XL", "Top-1"] - 1,
                    "Top-10 n1": dms_n1.loc["Flex (Duo)", "Top-10"] / dms_n1.loc["Explorer-XL", "Top-10"] - 1, "Top-10 n5": dms_n5.loc["Flex (Duo)", "Top-10"] / dms_n5.loc["Explorer-XL", "Top-10"] - 1}, index=["Flex (Duo) vs Explorer-XL"])
print("relative gain of Flex (Duo) over Explorer-XL:\n", (100 * rel).round(0).to_string(), "%   (the abstract says 25-50 % higher; the tables give 20 to 34 % against Explorer-XL)")

In [ ]:
# how often is the first-ranked route the one that is found at all?  P(Top-1 | Top-10)
share = pd.DataFrame({"n1": {**{k: base_n1.loc[k, "Top-1"] / base_n1.loc[k, "Top-10"] for k in base_n1.index},
                              "DMS Explorer-XL": .32 / .48, "DMS Flash": .41 / .60, "DMS Flex (Duo)": .43 / .61},
                      "n5": {**{k: base_n5.loc[k, "Top-1"] / base_n5.loc[k, "Top-10"] for k in base_n5.index},
                              "DMS Explorer-XL": .31 / .45, "DMS Flash": .37 / .53, "DMS Flex (Duo)": .39 / .54}})
print("Top-1 / Top-10: of the experimental routes found within 10 candidates, the share already ranked first")
print((100 * share).round(0).to_string(), "\n")
print("search baselines find routes at about the same rate within 10 candidates (Top-10 0.45-0.49 on n1) but rank them first only about a third of the time;")
print("DMS puts the experimental route first for about 70 % of those it finds. Most of its advantage is in RANKING, not in finding.")

fig, ax = plt.subplots(figsize=(6.6, 3.9))
solved = {"MCTS": (9714, .17), "Retro*": (9726, .15), "DFPN": (8475, .11), "DMS Explorer-XL": (8008, .32), "DMS Wide": (8089, .38), "DMS Flash": (8814, .41)}
for k, (sv, t1) in solved.items():
    ax.scatter(sv / 100, 100 * t1, s=45, color=COLORS["orange"] if k.startswith("DMS") else COLORS["blue"]); ax.annotate(k, (sv / 100, 100 * t1), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set(xlabel="targets solved (% of n1, stock = leaves of n1)", ylabel="Top-1 accuracy (%)"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("solving (reaching purchasable starting materials) and reproducing the experimental route are different things: the search methods solve more targets, DMS ranks the experimental route higher")

In [ ]:
# value of extra inputs: the same model with and without the starting material at generation (Top-1)
gain = pd.DataFrame({"n1": {"Flash": .41 / .32 - 1, "Flex (Duo)": .43 / .34 - 1, "Explorer": .36 / .27 - 1},
                     "n5": {"Flash": .37 / .30 - 1, "Flex (Duo)": .39 / .32 - 1, "Explorer": .33 / .25 - 1}})
print("relative Top-1 gain from giving the starting material at generation (%):"); print((100 * gain).round(0).to_string())
print("\nand the benefit of providing the number of steps: Explorer-XL .32 -> Deep .36 -> Wide .38 on n1 (no starting material in any of them)")

# search performance (typed): solved targets and run time per target, n1
t2 = pd.DataFrame({"solved": [8008, 8089, 8475, 8814, 9714, 9726], "time (s)": [14.7, 6.0, 347.3, 2.0, 303.3, 300.7]}, index=["DMS Explorer-XL", "DMS Wide", "DFPN", "DMS Flash", "MCTS", "Retro*"])
print("\n", t2.to_string())
print(f"time per target: MCTS / DMS Explorer-XL = {303.3 / 14.7:.0f}x. The comparison is not like for like: DMS ran on one A100 GPU with beam 50, the search methods are PaRoutes baselines with their own budgets.")
print("pipeline filters (Explorer-XL on n1): valid 9998 -> in-stock 8008 of 10,000 =", f"{8008 / 100:.1f} %; with the stock set changed from Buyables to eMolecules Screening on ChEMBL-5000: 68.66 % -> 75.58 %")

## 8. How long are the routes, and how sure can we be?
The paper reports that accuracy stays flat out to routes of 9 steps. The tables of accuracy by route length (S9 to S12) give accuracies as three-decimal fractions $k/n$; for *small* $n$ the denominator can be recovered. (For short routes, with hundreds of routes, rounding to three decimals does not determine $n$.) With the counts we can attach a confidence interval to each long-route accuracy.

In [ ]:
# Top-10 accuracy by route length 7..10, all eight models (Table S10 for n1, S12 for n5)
S10 = np.array([[.400, .333, .600, 0], [.323, .333, .600, 0], [.431, .333, .500, 0], [.400, .333, .500, 0], [.354, .333, .600, 0], [.231, .143, .300, 0], [.385, .333, .600, 0], [.338, .333, .600, 0]])
S9 = np.array([[.277, .286, .600, 0], [.262, .286, .600, 0], [.323, .286, .500, 0], [.308, .286, .400, 0], [.292, .286, .600, 0], [.154, .143, .300, 0], [.354, .286, .500, 0], [.277, .286, .400, 0]])
S12 = np.array([[.384, .476, .692, .333], [.368, .429, .615, .333], [.376, .405, .692, .333], [.400, .429, .692, .333], [.328, .476, .692, .333], [.272, .333, .462, .333], [.416, .500, .692, .333], [.400, .452, .692, .333]])
S11 = np.array([[.248, .405, .692, .333], [.272, .381, .385, .333], [.288, .286, .462, .333], [.328, .429, .692, .333], [.304, .381, .615, .333], [.184, .190, .154, .333], [.328, .476, .615, .333], [.344, .429, .615, .333]])

def smallest_denominator(vals, tol=0.00051, nmax=400):
    for n in range(1, nmax + 1):
        if all(abs(round(v * n) / n - v) <= tol for v in vals): return n
    return None

counts = {}
for name, tabs in (("n1", (S9, S10)), ("n5", (S11, S12))):
    A = np.vstack(tabs); counts[name] = [smallest_denominator(list(A[:, j])) for j in range(4)]
print("smallest route counts consistent with the tables, lengths 7, 8, 9, 10:", counts)
print("(Table S12 prints 0.47 for DMS-Explorer-XL at length 8 on n5; we read it as 0.476 = 20/42, the only reading consistent with the other entries of that column)")
print("check against the paper: 'there is only one route with length 10 in n1' ->", counts["n1"][3], "(our reconstruction)")

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z ** 2 / n; centre = (p + z ** 2 / (2 * n)) / den; half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return centre - half, centre + half

n_long = {"n1": dict(zip((7, 8, 9, 10), (65, 21, 10, 1))), "n5": dict(zip((7, 8, 9, 10), (125, 42, 13, 3)))}
rows = []
for ds, vals in (("n1", {9: 0.600, 8: 0.333}), ("n5", {9: 0.692, 8: 0.500})):                      # DMS-Flex (Duo), Top-10
    for L, acc in vals.items():
        n = n_long[ds][L]; k = round(acc * n); lo, hi = wilson(k, n); rows.append((ds, L, n, f"{k}/{n}", f"{acc:.3f}", f"{lo:.2f} to {hi:.2f}"))
print(pd.DataFrame(rows, columns=["test set", "route length", "routes", "found", "Top-10", "95 % interval"]).to_string(index=False))
print(f"\nthe 2-step routes: Top-10 {0.765:.3f} for n1 on several hundred routes, an interval of about +/- 0.03; the 9-step result rests on 10 (n1) or 13 (n5) routes")
print("so 'performance on 9-step routes is comparable to 2-step routes' is compatible with the data, but so is a much lower value: the interval for 6/10 is 0.31 to 0.83.")
print(f"routes with 7 or more steps: n1 {sum(n_long['n1'].values())} of 10,000 ({sum(n_long['n1'].values()) / 100:.1f} %), n5 {sum(n_long['n5'].values())} ({sum(n_long['n5'].values()) / 100:.1f} %)")

## 9. The search-space argument
The paper's motivation: a single-step model proposes $b$ candidate reactions per molecule, and a route with $n$ reactions can be assembled from them in about $b^n$ ways, so tree-search methods must be guided heavily. A direct route generator instead writes one string of about $L$ tokens with a beam of $B$ candidates.

In [ ]:
print(f"{'n reactions':>11} | {'routes if each step has b=10 options':>38} | {'tokens written (beam 50, ~' + str(round(tok_per_reaction)) + ' per reaction)':>44}")
for n in (2, 3, 4, 6, 9):
    print(f"{n:>11} | {10 ** n:>38,} | {round(50 * tok_per_reaction * n):>44,}")
print(f"(a rough illustration: ~{tok_per_reaction:.0f} tokens per reaction is read off our example route; the paper's longest route has 1074 tokens, about {1074 / tok_per_reaction:.0f} reactions' worth at this rate)")
print("The comparison hides a cost: a direct generator must have SEEN the right kind of route in training, while tree search can try anything its single-step model allows.")

## 10. Your turn
**Exercise 1 (trees).** Build a 3-step route with one node that has two non-leaf children. Compute its number of steps, leaves and whether it is convergent.

**Exercise 2 (strings).** How many distinct strings does your route have? Check with `permutations_of`.

**Exercise 3 (the score).** In Section 4 change `p_end` to 0.5 and to 0.05. How do the top-5 lengths change under the raw score and under the normalized score?

**Exercise 4 (experts).** Vary the number of hidden units `H` in the mixture-of-experts toy. Does the Mono penalty (ratio of the k=1 to the k=2 error) shrink when the experts are bigger?

**Exercise 5 (parameters).** Compute the parameter count of a DMS-Flash-like model with hidden size 512 and feed-forward multiplier 4 (the 60M experiments of the paper's Table S6).

**Exercise 6 (confidence intervals).** The paper states 'one route of length 10 in n1'. Why is `0.000` for all models consistent with that, and what would a Top-10 accuracy of 1.000 on that route tell you?

In [ ]:
# Starters
# Exercise 5
t, a = dms_params(8, 8, 1, 1, d=512, ff=4); print(f"hidden 512, FF x4, 8+8 layers: {t / 1e6:.1f} M parameters (Table S6 calls this 'Van-60', 60M)")

# Exercise 6
print("Wilson interval for 0 of 1:", tuple(round(x, 2) for x in wilson(0, 1)), "| for 1 of 1:", tuple(round(x, 2) for x in wilson(1, 1)))

## 11. Self-check

In [ ]:
assert n_steps(route) == 2 and n_reactions(route) == 2 and len(leaves(route)) == 3 and not convergent(route)
assert all(smiles_syntax_ok(s) for s in all_nodes(route)) and not smiles_syntax_ok("CC(C1CC")
assert ast.literal_eval(serialize(route)) == route and len(permutations_of(route)) == 4 and len(strings) == 4 and "".join(tokenize(serialize(route))) == serialize(route)
assert abs(inputs_noSM / routes_train - 2.643) < 1e-3 and abs(inputs_SM / inputs_noSM - 3.0975) < 1e-4
assert abs(np.mean([len(all_seq[i]) for i in np.argsort(-raw)[:5]]) - 1.0) < 1.01 and np.mean([len(all_seq[i]) for i in np.argsort(-norm)[:5]]) > 5
for name, (e, dcl, ne, ac, stated) in dms_variants.items():
    t, a = dms_params(e, dcl, ne, ac); target_m = {"10M": 10, "19M": 19, "50M": 50, "41M in Table 3 (38M in the text)": 41, "38M": 38}[stated]
    assert abs(t / 1e6 - target_m) / target_m < 0.06, name
assert dms_params(12, 36, 1, 1)[0] / 1e6 > 40 and abs(np.median(table[:, 0] / table[:, 1]) - 1.86) < 0.6 and (table[:, 0] > table[:, 1]).all() and np.allclose(table[:, 1], table[:, 2], rtol=0.05)
assert abs(lr_schedule(0.1 * T, T, 3.5e-4) - 3.5e-4) < 1e-12 and abs(lr_schedule(T, T, 3.5e-4) - 3.5e-5) < 1e-12 and abs(masked.mean() - 0.05) < 0.003
assert abs(dms_n1.loc["Explorer-XL", "Top-1"] / best_base_1 - 1.88) < 0.01 and abs(dms_n5.loc["Flex (Duo)", "Top-1"] / best_base_5 - 3.9) < 0.01
assert counts["n1"] == [65, 21, 10, 1] and counts["n5"] == [125, 42, 13, 3]
assert 0.30 < wilson(6, 10)[0] < 0.32 and 0.82 < wilson(6, 10)[1] < 0.84
print("All checks passed: you reproduced the numbers in the post.")